In [0]:
files = dbutils.fs.ls("dbfs:/mnt/packages")

for file in files:
    print(f"Name: {file.name}, Path: {file.path}, Size: {file.size}")

In [0]:
!pip install /dbfs/mnt/packages/*

In [0]:
from cubix_data_engineer_capstone.etl.bronze.extract_and_load_file import bronze_ingest

from cubix_data_engineer_capstone.etl.gold.wide_sales import get_wide_sales
from cubix_data_engineer_capstone.etl.gold.daily_sales_metrics import get_daily_sales_metrics
from cubix_data_engineer_capstone.etl.gold.daily_product_category_metrics import get_daily_product_category_metrics

from cubix_data_engineer_capstone.etl.silver.calendar import get_calendar
from cubix_data_engineer_capstone.etl.silver.customers import get_customers
from cubix_data_engineer_capstone.etl.silver.products import get_products
from cubix_data_engineer_capstone.etl.silver.product_subcategory import get_product_subcategory
from cubix_data_engineer_capstone.etl.silver.product_category import get_product_category
from cubix_data_engineer_capstone.etl.silver.sales import get_sales 
from cubix_data_engineer_capstone.etl.silver.scd import scd1

from cubix_data_engineer_capstone.utils.authentication import authenticate
from cubix_data_engineer_capstone.utils.datalake import read_file_from_datalake, write_file_to_datalake

In [0]:
authenticate()

#### Ingestion job

In [0]:
# The three batches of ingestion. Comment out the other two to use only one ingestion_task.

# Batch 2
# ingestion_task = {
#     "customers": {
#         "file_name": "customers_2.csv",
#         "primary_key": "CustomerKey"
#     },
#     "products": {
#         "file_name": "products_ingest_1.csv",
#         "primary_key": "ProductKey"
#     },
#     "product_subcategory": {
#         "file_name": "product_subcategory_ingest_1.csv",
#         "primary_key": "ProductSubcategoryKey"
#     },
#     "product_category": {
#         "file_name": "product_category_ingest_1.csv",
#         "primary_key": "ProductCategoryKey"
#     },
#     "sales": {
#         "file_name": "sales_202405.csv",
#         "primary_key": "SalesOrderNumber"
#     },
# }

# Batch 3
# ingestion_task = {
#     "customers": {
#         "file_name": "customers_4.csv",
#         "primary_key": "CustomerKey"
#     },
#     "sales": {
#         "file_name": "sales_202407.csv",
#         "primary_key": "SalesOrderNumber"
#     },
# }

# Batch 4
ingestion_task = {
    "customers": {
        "file_name": "customers_4.csv",
        "primary_key": "CustomerKey"
    },
    "sales": {
        "file_name": "sales_202407.csv",
        "primary_key": "SalesOrderNumber"
    },
}

#### Bronze layer

In [0]:
for dataset_key, params in ingestion_task.items():

    bronze_ingest(
        source_path=f"source_system/{dataset_key}",
        bronze_path=f"01_bronze/{dataset_key}",
        file_name=params["file_name"],
        container_name="capstoneproject",
        partition_by=None
    )

    print(f"{dataset_key} ({params['file_name']}) has been copied.")

#### Silver layer

In [0]:
transform_function_mapping = {
    "customers": get_customers,
    "sales": get_sales,
    "products": get_products,
    "product_category": get_product_category,
    "product_subcategory": get_product_subcategory
}

for dataset, params in ingestion_task.items():
    print(f"{dataset} is being processed.")
    
    raw_file = read_file_from_datalake(
        container_name="capstoneproject",
        file_path=f"01_bronze/{dataset}/{params['file_name']}",
        format="csv"
    )
    
    transform_function = transform_function_mapping.get(dataset)
    transformed_dataframe = transform_function(raw_file)

    # sales is not SCD, it's a Fact table, therefore appending.
    if dataset == "sales":
        write_file_to_datalake(
            df=transformed_dataframe,
            container_name="capstoneproject",
            file_path=f"02_silver/{dataset}",
            format="delta",
            mode="append",
            partition_by=None
        )
    else:
        scd1(spark, "capstoneproject", f"02_silver/{dataset}", transformed_dataframe, primary_key=params["primary_key"])

    print(f"{dataset} was updated.")

#### Gold layer

In [0]:
master_tables = [
    "sales",
    "calendar",
    "customers",
    "products",
    "product_category",
    "product_subcategory",
]

master_dataframes = {
    table: read_file_from_datalake(
        container_name="capstoneproject",
        file_path=f"02_silver/{table}",
        format="delta"
    )
    for table in master_tables
}

##### Wide Sales

In [0]:
wide_sales_df = get_wide_sales(
    sales_master=master_dataframes["sales"],
    customers_master=master_dataframes["customers"],
    products_master=master_dataframes["products"],
    product_category_master=master_dataframes["product_category"],
    product_subcategory_master=master_dataframes["product_subcategory"],
    calendar_master=master_dataframes["calendar"],
)

In [0]:
wide_sales_df.count()

In [0]:
display(wide_sales_df)

In [0]:
wide_sales_df.createOrReplaceTempView("wide_sales")

In [0]:
%sql

-- Changes in Customers
SELECT 
  DISTINCT
  CustomerKey,
  YearlyIncome,
  NumberChildrenAtHome,
  AddressLine1
FROM 
  wide_sales
WHERE 
  CustomerKey IN (11000, 11001)

In [0]:
%sql

-- Total sales and profit by month to identify high-performing months.
SELECT 
    MonthName, 
    CalendarYear,
    SUM(SalesAmount) AS TotalSales, 
    SUM(Profit) AS TotalProfit
FROM 
    wide_sales
GROUP BY 
    CalendarYear, MonthName, MonthNumberOfYear
ORDER BY 
    CalendarYear, MonthNumberOfYear;


In [0]:
%sql

-- Top high-value customers
SELECT 
    CustomerKey, 
    Name, 
    SUM(SalesAmount) AS TotalSales, 
    COUNT(SalesOrderNumber) AS TotalOrders
FROM 
    wide_sales
WHERE 
    HighValueOrder = true
GROUP BY 
    CustomerKey, Name
ORDER BY 
    TotalSales DESC
LIMIT 5;

In [0]:
%sql

-- Sales by customer demographics
SELECT 
    MaritalStatus, 
    Gender, 
    AVG(SalesAmount) AS AvgSales, 
    COUNT(SalesOrderNumber) AS TotalOrders
FROM 
    wide_sales
GROUP BY 
    MaritalStatus, Gender
ORDER BY 
    AvgSales DESC;

In [0]:
%sql

-- Which products are most often part of high-value orders.
SELECT 
    ProductName, 
    COUNT(SalesOrderNumber) AS HighValueOrderCount
FROM 
    wide_sales
WHERE 
    HighValueOrder = true
GROUP BY 
    ProductName
ORDER BY 
    HighValueOrderCount DESC;

In [0]:
write_file_to_datalake(wide_sales_df, "capstoneproject", "03_gold/wide_sales", "parquet")

##### Daily Sales metrics

In [0]:
daily_sales_metrics = get_daily_sales_metrics(wide_sales_df) 

In [0]:
display(daily_sales_metrics)

In [0]:
write_file_to_datalake(daily_sales_metrics, "capstoneproject", "03_gold/daily_sales_metrics", "parquet")

##### Daily Products metrics

In [0]:
daily_product_category_metrics = get_daily_product_category_metrics(wide_sales_df) 

In [0]:
display(daily_product_category_metrics)

In [0]:
write_file_to_datalake(daily_product_category_metrics, "capstoneproject", "03_gold/daily_product_category_metrics", "parquet")

#### Data Quality

In [0]:
!pip install great_expectations

In [0]:
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator
from great_expectations.execution_engine.sparkdf_execution_engine import SparkDFExecutionEngine
from great_expectations import get_context

In [0]:
# 1. Create a context:
context = get_context()

# 2. Create a Spark Execution Engine
execution_engine = SparkDFExecutionEngine()

# 3. Create a Batch from the DataFrame
batch = Batch(data=wide_sales_df)

# 4. Create a Validator with the batch and execution engine
validator = Validator(execution_engine=execution_engine, batches=[batch])

# 5. Add expectations
validator.expect_column_values_to_not_be_null(
    column="SalesOrderNumber", 
)

validator.expect_column_values_to_be_in_set(
    column="Gender", 
    value_set=["Male", "Female"])

validator.expect_column_values_to_be_between(
    column="BirthDate", 
    min_value="1999-01-01",
    max_value=None
)

# 6. Run validation and get results
results = validator.validate()

# 7. Process results
if results["success"]:
    print("All validations passed!")
else:
    print("Some validations failed.")
    for result in results["results"]:
        print(f"Expectation: {result['expectation_config']['type']}")
        print(f"Success: {result['success']}")
        if not result["success"]:
            print(f"Details: {result['result']}")


In [0]:
validation_results = results["results"]

results_data = [
    {
        "Expectation": res["expectation_config"]["type"],
        "Column": res["expectation_config"]["kwargs"].get("column"),
        "Success": res["success"],
        "Count": res["result"].get("element_count", "N/A"),
        "Failed Records Count": res["result"].get("unexpected_count", "N/A"),
        "Failed Records %": res["result"].get("unexpected_percent", "N/A"),
    }
    for res in validation_results
]

validation_results_df = spark.createDataFrame(results_data)

write_file_to_datalake(validation_results_df, "capstoneproject", "03_gold/wide_sales_validation_results", "parquet")

display(validation_results_df)